# Семинар 8. Оконные функции

На этом занятии вы будете нумеровать строки внутри групп, считать доли и накопительные
итоги, сравнивать соседние покупки и вычислять агрегаты в скользящей рамке.

**База.** Магазин цифровой музыки — схема `music_store` в учебной базе `hse_databases`.
Все задачи читают данные: создавать или изменять таблицы не нужно.
Инструкция по запуску PostgreSQL и Jupyter — в `README.md` в корне репозитория.

**Оценка.** Всего 6 задач на **10 баллов**, баллы указаны в заголовке каждой задачи.

В задачах 3–6 разбираем историю одного клиента: так результаты легко проверить вручную.

**Как работать.** Сначала выполните ячейку подключения и проверочный запрос.
Затем пишите решение каждой задачи в отдельной ячейке под условием. Первая строка `%%sql`
отправляет запрос в PostgreSQL и показывает результат таблицей.

## Данные для практики

Для всех задач достаточно таблицы **`music_store.invoice`**. Одна строка — счёт,
то есть одна покупка; в счёте может быть несколько треков. `total` — сумма всей покупки.

| Столбец | Что означает |
|---|---|
| `invoice_id` | Уникальный идентификатор счёта |
| `customer_id` | Идентификатор покупателя; у одного покупателя несколько счетов |
| `invoice_date` | Дата покупки, тип `date`, без времени |
| `billing_country` | Страна в счёте на момент покупки |
| `billing_city` | Город в счёте на момент покупки |
| `total` | Полная сумма счёта, денежные единицы |

Страна и город берутся **из счёта**, а не из текущей карточки покупателя.
Пишем имя таблицы вместе со схемой: `music_store.invoice`.

[Описание всех таблиц магазина](../db/schema.md#магазин-цифровой-музыки--схема-music_store).

![Схема магазина цифровой музыки](../db/music_store.svg)

In [ ]:
import sys; sys.path.insert(0, "../jupyter")
%load_ext sql_magic

### Проверка подключения
Если запрос ниже показал счета магазина, можно переходить к задачам.

In [ ]:
%%sql
SELECT
    invoice_id,
    customer_id,
    invoice_date,
    total
FROM music_store.invoice
ORDER BY invoice_id
LIMIT 5;

## Памятка

| Что нужно | Как пишется |
|---|---|
| Пронумеровать строки внутри группы | `ROW_NUMBER() OVER (PARTITION BY группа ORDER BY поле)` |
| Одинаковый номер равным значениям без пропусков | `DENSE_RANK() OVER (ORDER BY поле)` |
| Агрегат по разделу | `SUM(значение) OVER (PARTITION BY группа)` |
| Взять значение предыдущей строки | `LAG(значение) OVER (ORDER BY поле)` |
| Текущая и две предыдущие строки | `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` |
| Только пять предыдущих строк | `ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING` |
| От начала раздела до текущей строки | `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` |
| Начало недели или месяца | `DATE_TRUNC('week', дата)::date`, `DATE_TRUNC('month', дата)::date` |
| Округлить до двух знаков | `ROUND(значение, 2)` |

Неделя начинается в понедельник. Если отобрать покупки за календарный год, первая неделя
может начинаться ещё в декабре предыдущего года: покупки до начала выбранного года
при этом в расчёт не попадают.

`PARTITION BY` отделяет группы друг от друга. `ORDER BY` внутри `OVER` задаёт порядок
вычисления, а внешний `ORDER BY` — порядок строк в ответе.
Если у одного покупателя окажется несколько счетов одной даты, используйте
`ORDER BY invoice_date, invoice_id` для однозначного порядка отдельных покупок.

Номер из `ROW_NUMBER()` нельзя отфильтровать в `WHERE` того же уровня запроса:
сначала получите его в подзапросе или CTE, затем отфильтруйте во внешнем запросе.
Если нужно сначала сгруппировать данные, а затем применить к ним окно, можно также
выделить группировку в CTE.

Деление целых чисел отбрасывает дробную часть. Для процентов используйте `100.0`.
Чтобы не делить на ноль, используйте `CASE WHEN знаменатель IS NULL OR знаменатель = 0
THEN NULL ELSE ... END`. Округляйте итоговый процент, а не промежуточные значения.

На краю раздела рамка берёт доступные строки: среднее можно посчитать и по одной
или двум покупкам. Если в рамке вообще нет строк, `MAX` вернёт `NULL`.

### Задача 1. Первая покупка каждого клиента (1,5 балла)

Для каждого покупателя найдите его первую покупку с помощью ранжирующей оконной функции.
Если несколько покупок имеют одну дату, первой считайте покупку с меньшим `invoice_id`.

Выведите:

- **customer_id** — идентификатор покупателя;
- **invoice_id** — идентификатор первого счёта;
- **invoice_date** — дата первой покупки.

Отсортируйте по `customer_id` по возрастанию.

In [ ]:
%%sql
-- Задача 1. Напишите запрос

### Задача 2. Три самых дорогих уровня покупки (2 балла)

Магазин изучает крупные покупки каждой страны. Найдите счета, сумма которых
входит в **три наибольших различных значения** суммы в этой стране.
Если несколько счетов имеют одну сумму, сохраните их все.

Выведите **billing_country**, **invoice_id**, **invoice_date**, **total** и
**amount_rank** — место суммы среди различных сумм страны, начиная с 1.
Отсортируйте по стране, затем по сумме по убыванию, дате и номеру счёта.

После выполнения объясните: почему отбор первых трёх строк и отбор первых
трёх мест через `RANK()` могут дать другой набор счетов?

In [ ]:
%%sql
-- Задача 2. Напишите запрос

### Задача 3. Какая доля расходов приходится на покупку? (1,5 балла)

Для каждой покупки клиента **customer_id = 1** покажите её долю в общей сумме всех его покупок.
Общую сумму получите оконной функцией, сохранив отдельную строку для каждого счёта.

Выведите **invoice_id**, **invoice_date**, **total**, **customer_total** (сумма всех покупок клиента),
**share_pct** (доля текущей покупки в процентах, округлённая до двух знаков).
Отсортируйте по дате и номеру счёта.

Подсказка: окно без ORDER BY охватывает весь раздел.

In [ ]:
%%sql
-- Задача 3. Напишите запрос

### Задача 4. Как росли расходы клиента? (1,5 балла)

Для каждой покупки клиента **customer_id = 1** посчитайте, сколько он потратил
с начала своей истории до этой покупки включительно.

Выведите **invoice_id**, **invoice_date**, **total**, **cumulative_total** (накопленная сумма).
Порядок покупок — по дате, затем по номеру счёта. В таком же порядке выведите результат.

Подсказка: используйте SUM и рамку от UNBOUNDED PRECEDING до CURRENT ROW.

In [ ]:
%%sql
-- Задача 4. Напишите запрос

### Задача 5. Покупка стала дороже или дешевле? (2 балла)

Сравните каждую покупку клиента **customer_id = 1** с его предыдущей покупкой.

Выведите **invoice_id**, **invoice_date**, **total**, **previous_total** (сумма предыдущей покупки)
и **difference** (текущая сумма минус предыдущая).
Для первой покупки previous_total и difference должны быть NULL. Проценты считать не нужно.
Порядок истории и результата — по дате, затем по номеру счёта.

Подсказка: предыдущее значение получает LAG(total).

In [ ]:
%%sql
-- Задача 5. Напишите запрос

### Задача 6. Средняя сумма последних трёх покупок (1,5 балла)

Для каждой покупки клиента **customer_id = 1** найдите среднюю сумму текущей
и двух предыдущих покупок. Округлите среднее до двух знаков.
В начале истории считайте по доступным покупкам: для первой — по одной, для второй — по двум.

Выведите **invoice_id**, **invoice_date**, **total**, **rolling_avg_3** (скользящее среднее).
Порядок истории и результата — по дате, затем по номеру счёта.

Подсказка: используйте AVG и рамку ROWS BETWEEN 2 PRECEDING AND CURRENT ROW.
Сравните: чем эта рамка отличается от накопительного итога в задаче 4?

In [ ]:
%%sql
-- Задача 6. Напишите запрос